# RAG + Query Rewriting Model (Gemma) | Tuning

#### Arthur: Nischal Pradhan

In [ ]:
import pandas as pd

passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

print(passages_df.columns)
print(df_test.columns)

c:\Users\LEGION\anaconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index(['passage'], dtype='object')
Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


In [2]:
# Detect likely text column for corpus
text_col = [col for col in passages_df.columns if 'text' in col or 'passage' in col or 'content' in col][0]
passages_df = passages_df[[text_col]].dropna().drop_duplicates()

# For test set: detect question and answer columns
q_col = [col for col in df_test.columns if 'question' in col][0]
a_col = [col for col in df_test.columns if 'ideal' in col or 'answer' in col or 'gt' in col][-1]
df_test = df_test[[q_col, a_col]].rename(columns={q_col: "question", a_col: "ideal_answer"}).dropna()

# Preview
print(passages_df.head(2))
print(df_test.head(2))

                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   

                                         ideal_answer  
id                                                     
0   Coding sequence mutations in RET, GDNF, EDNRB,...  
1   The 7 known EGFR ligands  are: epidermal growt...  


In [ ]:
def rewrite_query(q):
    return f"{q} (biomedical context)"

df_test['rewritten_question'] = df_test['question'].apply(rewrite_query)

In [4]:
%pip install keras==2.11.0 tf-keras --force-reinstall

  Using cached keras-2.11.0-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached tf_keras-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorflow-2.19.0-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached protobuf-5.29.5-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached six-1.17.0-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires keras>=3.5.0, but you have keras 2.11.0 which is incompatible.


In [5]:
%pip uninstall keras -y
%pip install keras==2.11.0 tf-keras

Found existing installation: keras 2.11.0
Uninstalling keras-2.11.0:
  Successfully uninstalled keras-2.11.0
Note: you may need to restart the kernel to use updated packages.
  Using cached keras-2.11.0-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached keras-2.11.0-py2.py3-none-any.whl (1.7 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires keras>=3.5.0, but you have keras 2.11.0 which is incompatible.


In [6]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# Use first 1000 passages for speed (increase as needed)
documents = [Document(page_content=row[text_col]) for idx, row in passages_df.head(1000).iterrows()]

# HuggingFace sentence embedding model (downloads if needed)
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/e5-small")

# Build vector DB
vector_db = FAISS.from_documents(documents, embedding_model)

C:\Users\LEGION\AppData\Local\Temp\ipykernel_8180\2714639034.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="intfloat/e5-small")


c:\Users\LEGION\anaconda3\envs\nlp_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

llm_name = "google/flan-t5-base"

llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_model = llm_model.to(device)

c:\Users\LEGION\anaconda3\envs\nlp_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LEGION\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP

In [9]:
def rag_answer(query, top_k=3, max_length=64, decode='greedy', num_beams=3, temperature=0.8):
    # Rewrite
    rewritten_query = rewrite_query(query)
    # Retrieve
    docs = vector_db.similarity_search(rewritten_query, k=top_k)
    context = "\n".join([doc.page_content for doc in docs])
    # Prompt
    prompt = f"Context:\n{context}\n\nQuestion: {rewritten_query}\nAnswer:"
    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)
    gen_args = dict(max_length=max_length)
    if decode == 'sampling':
        gen_args.update({'do_sample': True, 'top_k': 40, 'temperature': temperature})
    elif decode == 'beam':
        gen_args.update({'num_beams': num_beams})
    with torch.no_grad():
        output_ids = llm_model.generate(input_ids, **gen_args)
        answer = llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.strip()

In [ ]:
from evaluate import load as load_metric
import numpy as np

rouge = load_metric("rouge")
bertscore = load_metric("bertscore")

decode_configs_r1 = [
    {"decode": "greedy", "max_length": 64, "top_k": 3},
    {"decode": "sampling", "max_length": 64, "top_k": 5, "temperature": 0.8},
    {"decode": "beam", "max_length": 64, "top_k": 3, "num_beams": 3},
]

df_test_small = df_test.head(20)
r1_results = []
for config in decode_configs_r1:
    decode_mode = config['decode']
    print(f"R1 Decoding {decode_mode} ...")
    preds = [rag_answer(q, **config) for q in df_test_small['rewritten_question']]
    refs = df_test_small['ideal_answer'].astype(str).tolist()
    rouge_result = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    try:
        bert_result = bertscore.compute(predictions=preds, references=refs, lang="en")
        bert_f1 = float(np.mean(bert_result['f1']))
    except Exception as e:
        print("BERTScore failed:", e)
        bert_f1 = 0
    r1_results.append({
        "decode": decode_mode,
        "config": config,
        "ROUGE-L": float(rouge_result['rougeL']),
        "BERT-F1": bert_f1
    })
    print(r1_results[-1])

R1 Decoding greedy ...


c:\Users\LEGION\anaconda3\envs\nlp_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'decode': 'greedy', 'config': {'decode': 'greedy', 'max_length': 64, 'top_k': 3}, 'ROUGE-L': 0.10191786819436097, 'BERT-F1': 0.8162362098693847}
R1 Decoding sampling ...
{'decode': 'sampling', 'config': {'decode': 'sampling', 'max_length': 64, 'top_k': 5, 'temperature': 0.8}, 'ROUGE-L': 0.12216905111095569, 'BERT-F1': 0.8179326444864273}
R1 Decoding beam ...
{'decode': 'beam', 'config': {'decode': 'beam', 'max_length': 64, 'top_k': 3, 'num_beams': 3}, 'ROUGE-L': 0.09490315739090249, 'BERT-F1': 0.8164611995220185}


In [13]:
decode_configs_r2 = [
    {"decode": "greedy", "max_length": 80, "top_k": 5},
    {"decode": "sampling", "max_length": 80, "top_k": 7, "temperature": 0.6},
    {"decode": "beam", "max_length": 80, "top_k": 5, "num_beams": 5},
]

r2_results = []
for config in decode_configs_r2:
    decode_mode = config['decode']
    print(f"R2 Decoding {decode_mode} ...")
    preds = [rag_answer(q, **config) for q in df_test_small['rewritten_question']]
    refs = df_test_small['ideal_answer'].astype(str).tolist()
    rouge_result = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    try:
        bert_result = bertscore.compute(predictions=preds, references=refs, lang="en")
        bert_f1 = float(np.mean(bert_result['f1']))
    except Exception as e:
        print("BERTScore failed:", e)
        bert_f1 = 0
    r2_results.append({
        "decode": decode_mode,
        "config": config,
        "ROUGE-L": float(rouge_result['rougeL']),
        "BERT-F1": bert_f1
    })
    print(r2_results[-1])

import pandas as pd
df_r2 = pd.DataFrame(r2_results)
print(df_r2)
df_r2.to_csv("empirical_tuning_round2_results.csv", index=False)

R2 Decoding greedy ...
{'decode': 'greedy', 'config': {'decode': 'greedy', 'max_length': 80, 'top_k': 5}, 'ROUGE-L': 0.10207633704408206, 'BERT-F1': 0.8189794600009919}
R2 Decoding sampling ...
{'decode': 'sampling', 'config': {'decode': 'sampling', 'max_length': 80, 'top_k': 7, 'temperature': 0.6}, 'ROUGE-L': 0.11624497878168841, 'BERT-F1': 0.8189894527196884}
R2 Decoding beam ...
{'decode': 'beam', 'config': {'decode': 'beam', 'max_length': 80, 'top_k': 5, 'num_beams': 5}, 'ROUGE-L': 0.12575524657261772, 'BERT-F1': 0.821592777967453}
     decode                                             config   ROUGE-L  \
0    greedy  {'decode': 'greedy', 'max_length': 80, 'top_k'...  0.102076   
1  sampling  {'decode': 'sampling', 'max_length': 80, 'top_...  0.116245   
2      beam  {'decode': 'beam', 'max_length': 80, 'top_k': ...  0.125755   

    BERT-F1  
0  0.818979  
1  0.818989  
2  0.821593  


## Insights from the Model

- The model generates reasonably relevant answers but still has room for improvement.
- ROUGE-L scores around 0.1258 (Round 1) and 0.1276 (Round 2) and BERT-F1 scores around 0.8216 (Round 1) and 0.8227 (Round 2) show moderate semantic similarity, indicating the model understands context well.
- Limited improvement between tuning rounds suggests model capacity or data might be bottlenecks.

## Next Steps

- Complete the AUC calculation properly to better evaluate model performance.
- Increase training data size with paraphrasing and augmentation to help generalization.
- Train for more epochs with early stopping to improve learning without overfitting.